# MSCI Filing Inspector
Structural analysis of the raw EDGAR HTML files before parsing.
Each cell answers one specific question about the data.

In [1]:
# Cell 1 — Setup & file paths
import os
from bs4 import BeautifulSoup

RAW_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'AI Projects', 'graph_rag', 'data', 'raw')
# If running from inside graph_rag/
RAW_DIR = os.path.join('data', 'raw')

FILES = {
    '10-K FY2025': os.path.join(RAW_DIR, 'msci_10k_fy2025.htm'),
    '10-Q Q1 2026': os.path.join(RAW_DIR, 'msci_10q_q1_2026.htm'),
}

for label, path in FILES.items():
    exists = os.path.exists(path)
    print(f'{label}: {path}  →  {"OK" if exists else "MISSING"}')

10-K FY2025: data\raw\msci_10k_fy2025.htm  →  OK
10-Q Q1 2026: data\raw\msci_10q_q1_2026.htm  →  OK


In [2]:
# Cell 2 — File sizes
print(f'{'File':<20} {'Size (MB)':>10} {'Raw chars':>12}')
print('-' * 45)

raw_html = {}
for label, path in FILES.items():
    size_mb = os.path.getsize(path) / 1_048_576
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    raw_html[label] = content
    print(f'{label:<20} {size_mb:>10.2f} {len(content):>12,}')

File                  Size (MB)    Raw chars
---------------------------------------------
10-K FY2025                2.61    2,737,314
10-Q Q1 2026               1.22    1,275,449


In [3]:
# Cell 3 — HTML element inventory
soups = {}
for label, html in raw_html.items():
    soups[label] = BeautifulSoup(html, 'html.parser')

print(f'{'Element':<12}', end='')
for label in FILES:
    print(f'{label:>15}', end='')
print()
print('-' * 42)

tags_to_check = ['table', 'img', 'svg', 'canvas', 'div', 'span', 'p', 'tr', 'td', 'th']
for tag in tags_to_check:
    print(f'{tag:<12}', end='')
    for label, soup in soups.items():
        count = len(soup.find_all(tag))
        print(f'{count:>15,}', end='')
    print()

Element         10-K FY2025   10-Q Q1 2026
------------------------------------------
table                    85             60
img                       2              1
svg                       0              0
canvas                    0              0
div                   2,395            890
span                  6,127          2,869
p                         0              0
tr                    1,283            851
td                   12,459          6,943
th                        0              0


In [4]:
# Cell 4 — Hidden XBRL block detection
# This is the display:none div containing machine-readable XBRL metadata.
# It is invisible in a browser but get_text() picks it up as garbage.

for label, soup in soups.items():
    print(f'=== {label} ===')
    hidden = soup.find('div', style=lambda s: s and 'display:none' in s)
    if hidden:
        text = hidden.get_text()
        print(f'  Hidden XBRL block : FOUND')
        print(f'  Char length       : {len(text):,}  ← will be stripped')
        print(f'  Sample (first 200 chars):')
        print(f'    {repr(text[:200])}')
    else:
        print('  Hidden XBRL block : NOT FOUND')
    print()

=== 10-K FY2025 ===
  Hidden XBRL block : FOUND
  Char length       : 23,927  ← will be stripped
  Sample (first 200 chars):
    '00014081982025FYFALSEP3Yhttp://fasb.org/us-gaap/2025#OtherAccruedLiabilitiesCurrentP1Yiso4217:USDxbrli:sharesiso4217:USDxbrli:sharesxbrli:puremsci:segment00014081982025-01-012025-12-3100014081982025-0'

=== 10-Q Q1 2026 ===
  Hidden XBRL block : FOUND
  Char length       : 12,632  ← will be stripped
  Sample (first 200 chars):
    '00014081982026Q112-31falsehttp://fasb.org/us-gaap/2025#OtherAccruedLiabilitiesCurrentxbrli:sharesiso4217:USDiso4217:USDxbrli:sharesxbrli:puremsci:Segment00014081982026-01-012026-03-3100014081982026-04'



In [5]:
# Cell 5 — Clean text after stripping hidden XBRL block
# Strip the hidden div, then measure what's left.

clean_soups = {}
for label, html in raw_html.items():
    s = BeautifulSoup(html, 'html.parser')
    hidden = s.find('div', style=lambda x: x and 'display:none' in x)
    if hidden:
        hidden.decompose()
    clean_soups[label] = s

print(f'{'File':<20} {'Clean chars':>12} {'Est. tokens':>12} {'Reduction':>10}')
print('-' * 58)
for label in FILES:
    raw_chars = len(raw_html[label])
    clean_text = clean_soups[label].get_text(separator=' ', strip=True)
    clean_chars = len(clean_text)
    est_tokens = clean_chars // 4
    reduction = raw_chars - clean_chars
    print(f'{label:<20} {clean_chars:>12,} {est_tokens:>12,} {reduction:>+10,}')

File                  Clean chars  Est. tokens  Reduction
----------------------------------------------------------
10-K FY2025               393,954       98,488 +2,343,360
10-Q Q1 2026              106,330       26,582 +1,169,119


In [6]:
# Cell 6 — Tables deep dive
# For every table: row count, column count, char length.
# Flags tables that exceed chunk size thresholds.

import statistics

for label, soup in clean_soups.items():
    tables = soup.find_all('table')
    lengths = [len(t.get_text()) for t in tables]

    print(f'=== {label} — {len(tables)} tables ===')
    print(f'  Min chars   : {min(lengths):,}')
    print(f'  Max chars   : {max(lengths):,}')
    print(f'  Avg chars   : {statistics.mean(lengths):,.0f}')
    print(f'  Median chars: {statistics.median(lengths):,.0f}')
    print(f'  > 500 chars : {sum(1 for l in lengths if l > 500)}')
    print(f'  > 1000 chars: {sum(1 for l in lengths if l > 1000)}  ← will be split across chunks')
    print(f'  > 2000 chars: {sum(1 for l in lengths if l > 2000)}  ← large financial tables')
    print(f'  > 3000 chars: {sum(1 for l in lengths if l > 3000)}')
    print()

    print(f'  {'#':<5} {'Rows':>5} {'Cells':>6} {'Chars':>7}  Preview')
    print(f'  ' + '-' * 70)
    for i, t in enumerate(tables):
        rows = len(t.find_all('tr'))
        cells = len(t.find_all(['td', 'th']))
        chars = len(t.get_text())
        preview = t.get_text()[:60].replace('\n', ' ').strip()
        flag = ' ◄ LARGE' if chars > 1000 else ''
        print(f'  {i:<5} {rows:>5} {cells:>6} {chars:>7,}  {preview}{flag}')
    print()

=== 10-K FY2025 — 85 tables ===
  Min chars   : 0
  Max chars   : 2,922
  Avg chars   : 599
  Median chars: 383
  > 500 chars : 31
  > 1000 chars: 14  ← will be split across chunks
  > 2000 chars: 4  ← large financial tables
  > 3000 chars: 0

  #      Rows  Cells   Chars  Preview
  ----------------------------------------------------------------------
  0         2      4       0  
  1         3     10     117  Delaware13-4038723(State or Other Jurisdiction ofIncorporati
  2         3     25     143  Title of each classTrading Symbol(s)Name of each exchange on
  3         4     24     114  Large accelerated filerxAccelerated fileroNon-accelerated fi
  4         2      4       0  
  5        31     99   1,190  PART IItem 1.Business2Item 1A.Risk Factors14Item 1B.Unresolv ◄ LARGE
  6        10     52     331  NameAgePositionHenry A. Fernandez67Chairman and Chief Execut
  7        12     83     433  LocationSquare FeetExpiration DateNew York, New York125,811(
  8         7     97     460 

In [7]:
# Cell 7 — Sample the 5 largest tables (full text)
# See exactly what financial tables look like when flattened to text.

for label, soup in clean_soups.items():
    tables = soup.find_all('table')
    sorted_tables = sorted(enumerate(tables), key=lambda x: len(x[1].get_text()), reverse=True)

    print(f'=== {label} — Top 5 largest tables ===')
    for rank, (idx, t) in enumerate(sorted_tables[:5]):
        text = t.get_text(separator=' | ', strip=True)
        rows = len(t.find_all('tr'))
        chars = len(t.get_text())
        print(f'\n--- Rank {rank+1}: table[{idx}]  ({rows} rows, {chars:,} chars) ---')
        print(text[:1000])
        if len(text) > 1000:
            print(f'  ... [{chars - 1000:,} more chars] ...')
    print()

=== 10-K FY2025 — Top 5 largest tables ===

--- Rank 1: table[38]  (67 rows, 2,922 chars) ---
Years Ended | (in thousands) | December 31, | 2025 | December 31, | 2024 | December 31, | 2023 | Cash flows from operating activities | Net income | $ | 1,202,305 | $ | 1,109,128 | $ | 1,148,592 | Adjustments to reconcile net income to net cash provided by operating activities: | Gain on remeasurement of equity method investment | — | — | ( | 143,029 | ) | Amortization of intangible assets | 169,480 | 164,037 | 114,429 | Stock-based compensation expense | 111,343 | 95,204 | 71,653 | Depreciation and amortization of property, equipment and leasehold improvements | 23,405 | 16,978 | 21,009 | Amortization of right of use assets | 25,685 | 25,260 | 23,781 | Loss on impairment of right of use assets, net | — | — | 477 | Amortization of debt origination fees | 5,876 | 5,143 | 5,055 | Loss on extinguishment of debt | — | 1,510 | — | Loss on investment in investee | 11,768 | — | — | Deferred taxes | 4

In [8]:
# Cell 8 — Images inventory
# Every image in the document: src, alt text, dimensions from style attribute.
# Decision: skip? OCR? relevant?

for label, soup in clean_soups.items():
    imgs = soup.find_all('img')
    print(f'=== {label} — {len(imgs)} images ===')
    if not imgs:
        print('  No images found.')
    for i, img in enumerate(imgs):
        src = img.get('src', 'N/A')
        alt = img.get('alt', 'N/A')
        style = img.get('style', 'N/A')
        print(f'  [{i}] src   : {src}')
        print(f'       alt   : {alt}')
        print(f'       style : {style}')
        # Guess if it's decorative or data
        is_logo = 'logo' in src.lower() or 'logo' in alt.lower()
        decision = 'SKIP (logo)' if is_logo else 'SKIP (decorative graphic — no financial data)'
        print(f'       decision: {decision}')
        print()
    print()

=== 10-K FY2025 — 2 images ===
  [0] src   : msci-20251231_g1.gif
       alt   : msci-logo-resized.gif
       style : height:30px;margin-bottom:5pt;vertical-align:text-bottom;width:112px
       decision: SKIP (logo)

  [1] src   : msci-20251231_g2.jpg
       alt   : 3554
       style : height:400px;margin-bottom:5pt;vertical-align:text-bottom;width:700px
       decision: SKIP (decorative graphic — no financial data)


=== 10-Q Q1 2026 — 1 images ===
  [0] src   : msci-20260331_g1.gif
       alt   : msci-logo-resized.gif
       style : height:30px;margin-bottom:5pt;vertical-align:text-bottom;width:112px
       decision: SKIP (logo)




In [9]:
# Cell 9 — Text quality samples
# Print 3 samples from different document sections to confirm text reads well.

for label, soup in clean_soups.items():
    full_text = soup.get_text(separator='\n', strip=True)
    print(f'=== {label} — Text quality samples ===')

    # Sample 1: Narrative / Business section
    idx = full_text.find('Item 1')
    if idx == -1:
        idx = 0
    print('--- Sample 1: Business section (Item 1 area) ---')
    print(full_text[idx:idx+600])
    print()

    # Sample 2: Financial data (Operating Revenue)
    idx = full_text.find('Operating Revenue')
    if idx == -1:
        idx = full_text.find('Operating revenue')
    print('--- Sample 2: Financial table area (Operating Revenue) ---')
    print(full_text[idx:idx+600])
    print()

    # Sample 3: Risk factors
    idx = full_text.find('Risk Factor')
    # Skip TOC mentions, find the actual section
    idx2 = full_text.find('Risk Factor', idx + 200)
    if idx2 != -1:
        idx = idx2
    print('--- Sample 3: Risk factors section ---')
    print(full_text[idx:idx+600])
    print()

=== 10-K FY2025 — Text quality samples ===
--- Sample 1: Business section (Item 1 area) ---
Item 1.
Business
2
Item 1A.
Risk Factors
14
Item 1B.
Unresolved Staff Comments
31
Item 1C.
Cybersecurity
31
Item 2.
Properties
33
Item 3.
Legal Proceedings
33
Item 4.
Mine Safety Disclosures
33
PART II
Item 5.
Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
34
Item 6.
[Reserved]
35
Item 7.
Management’s Discussion and Analysis of Financial Condition and Results of Operations
36
Item 7A.
Quantitative and Qualitative Disclosures About Market Risk
57
Item 8.
Financial Statements and Supplementary Data
59
Item 9.
Changes in and Disagreeme

--- Sample 2: Financial table area (Operating Revenue) ---
Operating Revenues
Our operating revenues are grouped by the following types: recurring subscriptions, asset-based fees and non-recurring. We also group operating revenues by major product as follows: Index, Analytics, Sustainability and Climate a

In [10]:
# Cell 10 — Chunking decision summary
# Based on all the above, print final recommended parameters for parse.py.

print('=' * 60)
print('CHUNKING DECISION SUMMARY')
print('=' * 60)

for label, soup in clean_soups.items():
    tables = soup.find_all('table')
    table_lengths = [len(t.get_text()) for t in tables]
    clean_text = soup.get_text(separator=' ', strip=True)
    total_chars = len(clean_text)
    large_tables = [(i, l) for i, l in enumerate(table_lengths) if l > 1000]

    print(f'\n--- {label} ---')
    print(f'  Total clean text        : {total_chars:,} chars')
    print(f'  Estimated tokens        : ~{total_chars // 4:,}')
    print(f'  Tables                  : {len(tables)}')
    print(f'  Tables > 1000 chars     : {len(large_tables)}  (indices: {[i for i,_ in large_tables]})')
    print()

print()
print('RECOMMENDED parse.py PARAMETERS')
print('-' * 60)
print('  chunk_size          = 1000  chars')
print('  chunk_overlap       = 100   chars')
print('  separators          = ["\\n\\n", "\\n", ". ", " "]')
print()
print('  XBRL hidden div     → strip before any text extraction')
print('  Images              → skip all (logo + decorative only)')
print('  Tables > 1000 chars → convert to pipe-delimited text first')
print('                        so column headers stay with values')
print('  Tables <= 1000 chars → keep as one chunk (fits in context)')
print()
print('  Metadata per chunk:')
print('    source, filing_type, period, chunk_index, chunk_type')

CHUNKING DECISION SUMMARY

--- 10-K FY2025 ---
  Total clean text        : 393,954 chars
  Estimated tokens        : ~98,488
  Tables                  : 85
  Tables > 1000 chars     : 14  (indices: [5, 28, 33, 34, 35, 37, 38, 48, 71, 75, 79, 80, 81, 82])


--- 10-Q Q1 2026 ---
  Total clean text        : 106,330 chars
  Estimated tokens        : ~26,582
  Tables                  : 60
  Tables > 1000 chars     : 6  (indices: [7, 8, 11, 12, 53, 58])


RECOMMENDED parse.py PARAMETERS
------------------------------------------------------------
  chunk_size          = 1000  chars
  chunk_overlap       = 100   chars
  separators          = ["\n\n", "\n", ". ", " "]

  XBRL hidden div     → strip before any text extraction
  Images              → skip all (logo + decorative only)
  Tables > 1000 chars → convert to pipe-delimited text first
                        so column headers stay with values
  Tables <= 1000 chars → keep as one chunk (fits in context)

  Metadata per chunk:
    source,